<div align="center">
  <h1>📈 Money Mentor 2.0</h1>
  <h3>End-to-End AI Financial Assistant & Quant Dashboard</h3>
</div>

**Money Mentor 2.0** is an advanced, multi-agent financial analysis platform. It bridges traditional quantitative finance (LSTM-based time-series forecasting) with modern Generative AI (Local LLMs, RAG, and LangGraph). 

Designed for scalability and memory efficiency on consumer hardware (GPU/Colab), this system autonomously fetches live market data, parses SEC filings, trains predictive models on the fly, and compiles comprehensive investment reports.

### 🏗️ System Architecture
1. **Data Layer:** Real-time ingestion via `yfinance`, web scraping (News), and SEC EDGAR API.
2. **Predictive Engine:** A Long Short-Term Memory (LSTM) neural network predicting price action using engineered technical indicators (RSI, MACD, Bollinger Bands).
3. **Retrieval-Augmented Generation (RAG):** Local vectorization of SEC filings and news using `BAAI/bge-small-en-v1.5` and `FAISS`.
4. **Multi-Agent Workflow:** `LangGraph` orchestrates distinct agents (Prediction, Fundamentals, SEC, News, RAG, Advisor, Explainability, Evaluation).
5. **Generative UI:** `Qwen2.5-3B-Instruct` (4-bit quantized) powers the conversational memory and report generation, visualized through a `Gradio` dashboard.

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
import subprocess
import sys
import os

def install_dependencies():
    print("Installing dependencies... This may take a few minutes.")
    packages = [
        "yfinance", "gradio", "reportlab", "langgraph", "langchain-core",
        "faiss-cpu", "sentence-transformers", "beautifulsoup4", "requests",
        "plotly", "shap", "transformers", "accelerate", "bitsandbytes", "tf-keras"
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U"] + packages, check=True)
    print("Dependencies installed successfully!")

if not os.path.exists("/content/installed.flag"):
    install_dependencies()
    open("/content/installed.flag", "w").close()

### 📦 Imports & System Configuration
We utilize a robust caching system to prevent API rate-limiting from Yahoo Finance and the SEC.

In [ ]:
# ==============================================================================
# 2. IMPORTS & CACHING SYSTEM
# ==============================================================================
import json
import time
import datetime
import traceback
import numpy as np
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import gradio as gr
import plotly.graph_objects as go
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.utils import simpleSplit
import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from sklearn.preprocessing import MinMaxScaler
from typing import Dict, List, TypedDict, Any
from langgraph.graph import StateGraph, END

CACHE_DIRS = ["cache", "cache/yfinance", "cache/sec", "cache/news", "cache/embeddings", "cache/vectorstore", "cache/reports", "cache/models"]
for d in CACHE_DIRS:
    os.makedirs(d, exist_ok=True)

SEC_HEADERS = {"User-Agent": "MoneyMentor Admin admin@moneymentor.ai"}

def get_cache(filename, max_age_hours=24):
    filepath = os.path.join("cache", filename)
    if os.path.exists(filepath):
        if (time.time() - os.path.getmtime(filepath)) < max_age_hours * 3600:
            with open(filepath, 'r') as f:
                return json.load(f)
    return None

def save_cache(filename, data):
    filepath = os.path.join("cache", filename)
    with open(filepath, 'w') as f:
        json.dump(data, f)

### 🧠 Local LLM & Vector Store Initialization
To maintain data privacy and eliminate API costs, the system uses **Qwen2.5-3B-Instruct** loaded locally with 4-bit quantization.

In [ ]:
# ==============================================================================
# 3. LLM & EMBEDDINGS INIT
# ==============================================================================
print("Loading Local Models...")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16) if DEVICE == "cuda" else None
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
llm_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", quantization_config=quant_config, device_map="auto" if DEVICE == "cuda" else None, low_cpu_mem_usage=True)
llm_pipeline = pipeline("text-generation", model=llm_model, tokenizer=tokenizer)
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
faiss_dim = embedder.get_sentence_embedding_dimension()
vector_db = faiss.IndexFlatL2(faiss_dim)
rag_documents_store = []

def ask_llm(system_prompt, user_prompt, max_tokens=256):
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm_pipeline(formatted, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return out[0]['generated_text'][len(formatted):].strip()

### 📊 Data Pipeline & Feature Engineering
Engineers technical indicators (MA, EMA, RSI, MACD, Bollinger Bands) to provide the LSTM network with rich context.

In [ ]:
# ==============================================================================
# 4. DATA COLLECTION & FEATURE ENGINEERING
# ==============================================================================
def fetch_stock_data(ticker):
    df = yf.download(ticker, period="2y", interval="1d", progress=False)
    if df.empty: raise ValueError(f"No data found for {ticker}")
    df['MA_50'] = df['Close'].rolling(window=50).mean()
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    df['MACD'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    df['BB_Upper'] = df['MA_50'] + 2 * df['Close'].rolling(window=20).std()
    df['BB_Lower'] = df['MA_50'] - 2 * df['Close'].rolling(window=20).std()
    df.bfill(inplace=True)
    return df

def fetch_fundamentals(ticker):
    cache_key = f"yfinance/{ticker}_fundamentals.json"
    cached = get_cache(cache_key)
    if cached: return cached
    info = yf.Ticker(ticker).info
    data = {"Market Cap": info.get("marketCap", "N/A"), "PE Ratio": info.get("trailingPE", "N/A"), "Recommendation": info.get("recommendationKey", "N/A")}
    save_cache(cache_key, data)
    return data

def fetch_sec_filings(ticker):
    cache_key = f"sec/{ticker}_sec.json"
    cached = get_cache(cache_key)
    if cached: return cached
    try:
        tickers_json = requests.get("https://www.sec.gov/files/company_tickers.json", headers=SEC_HEADERS).json()
        cik = next((str(v['cik_str']).zfill(10) for k, v in tickers_json.items() if v['ticker'].upper() == ticker.upper()), None)
        if not cik: return {"summary": "CIK not found for SEC filings."}
        subs = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=SEC_HEADERS).json()
        filings = subs['filings']['recent']
        summary = "Recent Filings:\n" + "\n".join([f"- {form} on {filings['filingDate'][i]}" for i, form in enumerate(filings['form']) if form in ["10-K", "10-Q"]][:3])
        data = {"summary": summary}
        save_cache(cache_key, data)
        return data
    except Exception as e:
        return {"summary": f"Failed to retrieve SEC data: {str(e)}"}

def fetch_news(ticker):
    try:
        news_data = yf.Ticker(ticker).news[:5]
        return {"headlines": [f"{n['title']}" for n in news_data]}
    except: return {"headlines": ["No recent news available."]}

### 🔮 Deep Learning: LSTM Time-Series Forecasting
Trains a sequence-to-vector LSTM model on a rolling 60-day window.

In [ ]:
# ==============================================================================
# 5. LSTM PREDICTION MODULE
# ==============================================================================
def train_or_load_lstm(ticker, df):
    model_path = f"cache/models/{ticker}_lstm.keras"
    features = ['Close', 'Volume', 'MA_50', 'RSI', 'MACD']
    data = df[features].values
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data)
    seq_length = 60
    X, y = np.array([scaled_data[i-seq_length:i] for i in range(seq_length, len(scaled_data))]), np.array([scaled_data[i, 0] for i in range(seq_length, len(scaled_data))])
    if os.path.exists(model_path):
        model = load_model(model_path)
    else:
        model = Sequential([Input(shape=(X.shape[1], X.shape[2])), LSTM(50, return_sequences=True), Dropout(0.2), LSTM(50, return_sequences=False), Dropout(0.2), Dense(25), Dense(1)])
        model.compile(optimizer='adam', loss='mean_squared_error')
        model.fit(X, y, batch_size=32, epochs=5, verbose=0)
        model.save(model_path)
    last_60 = np.expand_dims(scaled_data[-seq_length:], axis=0)
    pred_scaled = model.predict(last_60, verbose=0)
    dummy = np.zeros((1, len(features)))
    dummy[0, 0] = pred_scaled[0, 0]
    predicted_price = scaler.inverse_transform(dummy)[0, 0]
    current_price = df['Close'].iloc[-1]
    pct_change = ((predicted_price - current_price) / current_price) * 100
    return {"current_price": float(current_price), "predicted_price": float(predicted_price), "change_percent": float(pct_change), "trend": "Bullish" if pct_change > 0 else "Bearish"}

### 🕸️ LangGraph: Multi-Agent Orchestration & UI
State graph orchestration and Gradio frontend deployment.

In [ ]:
# ==============================================================================
# 6. LANGGRAPH ORCHESTRATION & GRADIO UI
# ==============================================================================
class MoneyMentorState(TypedDict):
    ticker: str
    stock_data: Any
    prediction: dict
    fundamentals: dict
    sec_data: dict
    news_data: dict
    investment_report: str
    logs: list

def log_agent(state, name, msg):
    if 'logs' not in state: state['logs'] = []
    state['logs'].append(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {name}: {msg}")
    return state

def prediction_agent(state): 
    state = log_agent(state, "Quant", "Running LSTM...")
    state["stock_data"] = fetch_stock_data(state["ticker"])
    state["prediction"] = train_or_load_lstm(state["ticker"], state["stock_data"])
    return state

def data_agent(state):
    state = log_agent(state, "Data", "Fetching Fundamentals, SEC, News...")
    state["fundamentals"] = fetch_fundamentals(state["ticker"])
    state["sec_data"] = fetch_sec_filings(state["ticker"])
    state["news_data"] = fetch_news(state["ticker"])
    return state

def advisor_agent(state):
    state = log_agent(state, "Advisor", "Generating AI Report...")
    prompt = f"Analyze {state['ticker']}. Predict: ${state['prediction']['predicted_price']:.2f}. News: {state['news_data']['headlines']}. Give a concise summary."
    state["investment_report"] = ask_llm("You are a Quant Advisor.", prompt, 250)
    return state

workflow = StateGraph(MoneyMentorState)
workflow.add_node("Predict", prediction_agent)
workflow.add_node("Data", data_agent)
workflow.add_node("Advisor", advisor_agent)
workflow.set_entry_point("Predict")
workflow.add_edge("Predict", "Data")
workflow.add_edge("Data", "Advisor")
workflow.add_edge("Advisor", END)
graph = workflow.compile()

def process(ticker):
    try:
        state = graph.invoke({"ticker": ticker.upper(), "logs": []})
        return f"### Predict: ${state['prediction']['predicted_price']:.2f}", state["investment_report"], "\n".join(state["logs"])
    except Exception as e: return str(e), "", ""

with gr.Blocks() as ui:
    gr.Markdown("# 📈 Money Mentor 2.0")
    t_input = gr.Textbox(label="Ticker")
    btn = gr.Button("Analyze")
    pred, rep, logs = gr.Markdown(), gr.Markdown(), gr.Textbox(label="Logs")
    btn.click(process, inputs=[t_input], outputs=[pred, rep, logs])

ui.launch(debug=True, share=True)